In [ ]:
import marimo as mo

# 🧪 LAB: Implement an autoencoder with `PyTorch`

In this lab, you will implement and explore autoencoders using the MNIST dataset. The goal is to understand how autoencoders learn compressed representations of data, and how these representations compare with classical methods like PCA.

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

---

In total, this lab assignment will be worth **100 points**.

---
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.

In [ ]:
NAMES = "Sae-Jin Moon\ngroup11-section1"

## 1. Theoretical Background Reflection (10 points)

Read *Michelucci (2022)*: [An Introduction to Autoencoders](https://arxiv.org/abs/2201.03898) — focus on the following sections:
- 1. Introduction
- 2.1, 2.2, and 2.3
- Skim Section 3 on applications

Then answer briefly:

1. What is the main idea behind an autoencoder?
2. What is the role of the bottleneck layer?
3. Discuss, in your own words, the applications of autoencoders mentioned in the paper.
4. What differences would you expect between an autoencoder’s representation and that of PCA if applied as a dimensionality reduction technique?

### Theoretical Background Answers

#### 1. Main idea behind an autoencoder
An autoencoder is a neural network trained to reconstruct its own input without needing manual labels. The model passes an input observation $x$ through an encoder function to generate a compressed internal code, and then passes that code through a decoder function to produce an output $\tilde{x}$. Training adjusts the network weights so that the difference between $x$ and $\tilde{x}$ stays as small as possible. Because the network cannot simply copy the input when constrained by a smaller intermediate layer, it learns the statistical structure of the training data.

#### 2. Role of the bottleneck layer
The bottleneck layer sits in the middle of the network and contains fewer neurons than the original input ($q < n$). This size restriction creates an information bottleneck that stops the network from learning a simple identity shortcut. By forcing the input through this narrow layer, the encoder discards noise and redundant pixel values while keeping the essential coordinates the decoder needs to reconstruct the original image.

#### 3. Applications discussed in Michelucci (2022)
Michelucci reviews four key applications in Section 3:
- **Dimensionality reduction**: The encoder compresses high-dimensional inputs into $q$ continuous values, giving a compact code for plotting or storage.
- **Classification with latent features**: Downstream classifiers run much faster on compressed coordinates. Michelucci shows that running $k$-nearest neighbors on 60,000 MNIST digits took 1 hour and 54 minutes on raw 784-dimensional pixels, but only 22 seconds on 32-dimensional autoencoder latents, which is roughly a 300-fold speedup with virtually no drop in classification accuracy.
- **Anomaly detection**: The autoencoder trains only on normal data and learns low reconstruction error on normal samples. When an unusual sample comes in, the network cannot reconstruct it well, producing a large reconstruction error that flags the anomaly.
- **Denoising**: The network takes inputs with added noise (like random Gaussian noise) and trains against clean target images. This teaches the autoencoder to strip out background noise and recover the underlying image.

#### 4. Differences between an autoencoder representation and PCA
PCA is a linear transformation that projects data onto the eigenvectors of the dataset covariance matrix to maximize variance. Because it is strictly linear, PCA cannot bend coordinate axes to follow curved shapes in the data. PCA also requires $O(d^2)$ memory and full-dataset matrix computations, which gets expensive on very large feature spaces. In contrast, an autoencoder with non-linear activation functions (like ReLU) can learn non-linear manifolds and resolve curved relationships that PCA collapses. Autoencoders also train with mini-batch gradient descent, letting them scale to large datasets where full-batch PCA runs out of memory.

## 2. Prepare the Data (10 points)

Given the below dataset:

- Normalize **each** image (i.e., separately) so its values are between 0 and 1. Remember our first-week discussions of fixed and fitted transformations.
- Visualize a few random digits to confirm everything worked as expected.
- Split the dataset into fitting, tuning and reporting sets (70%, 15%, 15%).
- Wrap each data splitting as `DataLoader` object with a batch size of 128. Call them `fitting_loader`, `tuning_loader` and `reporting_loader`, respectively.

### Fixed vs Fitted Transformations Note
Min-max scaling applied per image computes $(x_i - \min(x_i)) / (\max(x_i) - \min(x_i))$ strictly within each 784-pixel image. Because each image is scaled independently using only its own pixel values, this transformation is fixed and sample-wise. It does not estimate dataset-wide parameters (such as global mean or standard deviation across images), so applying it across the full dataset before splitting does not cause data leakage between fitting, tuning, and reporting sets.

In [ ]:
from sklearn.datasets import fetch_openml

# load the raw mnist dataset
mnist = fetch_openml("mnist_784", version=1, as_frame=False, n_retries=5, delay=2)
X, y = mnist.data, mnist.target.astype(int)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

# fix random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# convert to float32
x_raw = X.astype(np.float32)

# sample-wise min-max normalization per image
row_mins = x_raw.min(axis=1, keepdims=True)
row_maxs = x_raw.max(axis=1, keepdims=True)
x_norm = (x_raw - row_mins) / np.maximum(row_maxs - row_mins, 1e-8)

# visualize 10 sample digits to verify normalization
_fig, _axes = plt.subplots(2, 5, figsize=(10, 4.2))
sample_indices = np.random.choice(len(x_norm), size=10, replace=False)
for idx, ax in zip(sample_indices, _axes.ravel()):
    ax.imshow(x_norm[idx].reshape(28, 28), cmap="gray_r")
    ax.set_title(f"digit: {y[idx]}")
    ax.axis("off")
plt.suptitle("sample normalized mnist digits", fontsize=12)
plt.tight_layout()
plt.show()

# split into fitting (70%), tuning (15%), and reporting (15%) sets
x_fit, x_temp, y_fit, y_temp = train_test_split(
    x_norm, y, test_size=0.30, random_state=42, stratify=y
)
x_tune, x_report, y_tune, y_report = train_test_split(
    x_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# create dataloaders with batch size 128
fitting_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_fit), torch.from_numpy(y_fit)),
    batch_size=128,
    shuffle=True,
)
tuning_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_tune), torch.from_numpy(y_tune)),
    batch_size=128,
    shuffle=False,
)
reporting_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_report), torch.from_numpy(y_report)),
    batch_size=128,
    shuffle=False,
)

## 3. Autoencoder

In this section, you will build and train an **autoencoder** to reconstruct the provided data.

Your model will include three layers in both the encoder and decoder:

- **Encoder:** 784 → 128 → 64 → 2. This last layer here is the **bottleneck**.
- **Decoder:** 2 → 64 → 128 → 784

### 3.1 Implementation (30 points)

Follow the skeleton provided below and fill in the missing parts (`#TODO`).

Follow these restrictions:

- Each **hidden** layer should include a linear transformation, ReLU activation function, dropout with `p=0.2` and batch normalization steps. Determine the appropriate order for these components based on what we discussed in class. The bottleneck layer is the exception; see next point.
- For the bottleneck, use only a linear transformation, i.e., no activation, batch normalization, or dropout.
- The output layer should have a sigmoid activation function.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        # encoding part: 784 -> 128 -> 64 -> 2
        # hidden layers use linear -> batchnorm1d -> relu -> dropout(0.2)
        # bottleneck layer uses only a linear transformation
        encoder_list = [
            nn.Linear(784, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2),
        ]
        self.encoder_list = nn.ModuleList(encoder_list)

        # decoding part: 2 -> 64 -> 128 -> 784
        # hidden layers use linear -> batchnorm1d -> relu -> dropout(0.2)
        # output layer uses linear followed by sigmoid
        decoder_list = [
            nn.Linear(2, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 784),
            nn.Sigmoid(),
        ]
        self.decoder_list = nn.ModuleList(decoder_list)

    def forward(self, X):
        for layer in self.encoder_list:
            X = layer(X)
        for layer in self.decoder_list:
            X = layer(X)
        return X

### 3.2 Training (15 points)

Follow the skeleton provided below and fill in the missing parts to train your model (`#TODO`). As you can see, training includes EarlyStopping, which I give you already for your convenience :-). After you have filled these missing parts, uncomment the provided piece of code with the function for training (`train_autoencoder`) your model and execute it.

But, **BEFORE** that, discuss with your mates and provide elaborated answers to the following:

- Why do you think the output layer of the autoencoder above uses a Sigmoid activation?
- Based on this, which cost function is most appropriate for this task? Justify your choice.

### Output Activation and Cost Function Discussion

#### Why Sigmoid activation at the output?
In Section 2, we normalized every image so that pixel intensities fall strictly into the interval $[0, 1]$. The sigmoid activation function maps real numbers to outputs between 0 and 1. Placing sigmoid at the output ensures that every reconstructed pixel is bounded within the valid input range, preventing negative pixel intensities or values above 1.

#### Which cost function is most appropriate?
Binary cross-entropy (`nn.BCELoss`) is the most appropriate cost function. With normalized pixel intensities in $[0, 1]$, each pixel can be treated as a Bernoulli probability indicating how dark that pixel is. Binary cross-entropy evaluates pixel-by-pixel cross-entropy across all 784 dimensions. Pairing binary cross-entropy with a sigmoid output layer also keeps gradient updates stable, avoiding the vanishing gradients that mean squared error encounters near 0 or 1.

Once you have discussed and agreed on the reasoning above, proceed to complete the skeleton (`#TODO`) and train the model.

In [ ]:
# check hardware accelerator
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# instantiate model on device
model = Autoencoder().to(device)

# binary cross-entropy loss for pixels scaled [0, 1]
criterion = nn.BCELoss()

# adam optimizer with lr 0.001 and weight decay 1e-5
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, tuning_loss, model):
        if self.best_loss is None or tuning_loss < self.best_loss - self.min_delta:
            self.best_loss = tuning_loss
            self.counter = 0
            self.best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

    def restore_best_weights(self, model):
        if self.best_model_state is not None:
            model.load_state_dict(self.best_model_state)

def train_autoencoder(model, fitting_loader, tuning_loader, n_epochs=100, patience=5):
    # initialize the early stopping object
    early_stopping = EarlyStopping(patience=patience)
    history = {"fitting": [], "tuning": []}
    dev = next(model.parameters()).device

    for epoch in range(n_epochs):
        model.train()
        fitting_cost = 0.0

        for X_batch, _ in fitting_loader:
            # forward pass for this batch
            X_batch = X_batch.to(dev)
            outputs = model(X_batch)
            cost = criterion(outputs, X_batch)

            # backward pass and parameter update
            cost.backward()
            optimizer.step()
            optimizer.zero_grad()

            fitting_cost += cost.item() * X_batch.size(0)

        fitting_cost /= len(fitting_loader.dataset)

        # tuning phase
        model.eval()
        tuning_cost = 0.0
        with torch.no_grad():
            for X_batch, _ in tuning_loader:
                X_batch = X_batch.to(dev)
                outputs = model(X_batch)
                cost = criterion(outputs, X_batch)
                tuning_cost += cost.item() * X_batch.size(0)

        tuning_cost /= len(tuning_loader.dataset)

        # save metrics
        history["fitting"].append(fitting_cost)
        history["tuning"].append(tuning_cost)

        print(f"Epoch [{epoch+1}/{n_epochs}]  Fitting Cost: {fitting_cost:.5f}  Tuning Cost: {tuning_cost:.5f}")

        # early stopping check
        early_stopping(tuning_cost, model)
        if early_stopping.early_stop:
            print(f"Stopping early at epoch {epoch+1}")
            early_stopping.restore_best_weights(model)
            return model, history

    return model, history

Once the missing parts have been filled, uncomment the piece of code below and execute it to train your model (it may take a bit to complete the training, which is totally normal).

In [ ]:
# train autoencoder with early stopping
_, history = train_autoencoder(model, fitting_loader, tuning_loader, n_epochs=50, patience=5)

# plot training and tuning loss curves
plt.figure(figsize=(8, 4))
epochs_range = range(1, len(history["fitting"]) + 1)
plt.plot(epochs_range, history["fitting"], label="fitting loss", color="royalblue", lw=1.8)
plt.plot(epochs_range, history["tuning"], label="tuning loss", color="darkorange", lw=1.8)
plt.xlabel("epoch")
plt.ylabel("bce loss")
plt.title("autoencoder training and tuning cost curves")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.3 Reconstruct (5 points)

Once you have trained the model:

1. Visualize several reconstructed digits from the reporting set.
2. Compare them with the corresponding original input digits. Was the reconstruction good?

In [ ]:
# extract one batch from reporting loader
model.eval()
dev = next(model.parameters()).device
with torch.no_grad():
    x_batch_rep, y_batch_rep = next(iter(reporting_loader))
    x_batch_dev = x_batch_rep.to(dev)
    recon_batch = model(x_batch_dev).cpu()

# plot 10 original digits and their reconstructions side-by-side
_fig, _axes = plt.subplots(2, 10, figsize=(15, 3.5))
for i in range(10):
    # original digit
    _axes[0, i].imshow(x_batch_rep[i].reshape(28, 28), cmap="gray_r")
    _axes[0, i].set_title(f"orig: {y_batch_rep[i].item()}", fontsize=9)
    _axes[0, i].axis("off")

    # reconstructed digit
    _axes[1, i].imshow(recon_batch[i].reshape(28, 28), cmap="gray_r")
    _axes[1, i].set_title("recon", fontsize=9)
    _axes[1, i].axis("off")

_axes[0, 0].set_ylabel("original", fontsize=10)
_axes[1, 0].set_ylabel("reconstructed", fontsize=10)
plt.suptitle("reporting set: original vs. reconstructed digits", fontsize=12)
plt.tight_layout()
plt.show()

### Reconstruction Quality Evaluation
The reconstructions capture the broad geometry and loop shapes of the digits, but they are visibly blurry and smooth out sharp strokes. Digits with simple outlines like 0 and 1 remain recognizable. Digits with finer internal lines or similar shapes (such as 3 and 8, or 4 and 9) get muddier and lose edge definition.

This blurriness is expected because the bottleneck compresses 784 input pixels down to just 2 continuous dimensions. Compressing the image by a factor of 392 forces the network to drop fine stroke variations and keep only the coarse global layout needed to minimize average binary cross-entropy.

## 4. Visualize the Learned Representations (15 points)

Here you will explore the latent representations learned by your autoencoder.

1. Complete `extract_latent_features` to extract the latent (bottleneck) features and their corresponding labels, given a trained model.
   Apply this function to the reporting set using the provided code. This should give you the transformed reporting set with the dimensions of the bottleneck (2).
2. Plot the reduced 2D data as a scatterplot, coloring each point by its corresponding digit label.

In [ ]:
def extract_latent_features(model, dataloader):
    model.eval()
    latents = []
    labels = []
    dev = next(model.parameters()).device

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            # pass inputs through encoder layers to bottleneck
            encoded = X_batch.to(dev)
            for layer in model.encoder_list:
                encoded = layer(encoded)
            latents.append(encoded.cpu().numpy())
            labels.append(y_batch.numpy())

    # concatenate all batches
    latents = np.concatenate(latents, axis=0)
    labels = np.concatenate(labels, axis=0)
    return latents, labels

Once the missing parts above have been filled, uncomment the piece of code below and execute it to extract the latent features

In [ ]:
# extract 2d latent features on reporting set
latents_autoencoder, labels = extract_latent_features(model, reporting_loader)

# scatter plot of autoencoder 2d latent space
plt.figure(figsize=(9, 7))
scatter = plt.scatter(
    latents_autoencoder[:, 0],
    latents_autoencoder[:, 1],
    c=labels,
    cmap="tab10",
    alpha=0.55,
    s=8,
)
cbar = plt.colorbar(scatter, ticks=range(10))
cbar.set_label("digit label", fontsize=10)
plt.xlabel("latent dimension 1")
plt.ylabel("latent dimension 2")
plt.title("autoencoder 2d latent space (reporting set)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Latent Space Analysis
The 2D latent plot organizes digits into distinct curved bands and clusters. Digits with simple topologies like 0 and 1 form well-separated clusters on opposite sides of the space, whereas overlapping digits like 3 and 5 or 4 and 9 crowd together near the middle. Squeezing 10 digit classes into only 2 dimensions forces visually similar numbers to share adjacent coordinates in the bottleneck layer.

## 5. Compare with PCA (10 points)

Here you will compare the latent representations learned by your autoencoder with those obtained using a PCA.

1. Apply a PCA to the reporting set (without using the autoencoder) to reduce the data to the same dimensionality as your bottleneck (2 dimensions). Obviously, first use the rest of the data to learn these two components.
2. Visualize the reduced 2D data as a scatterplot, coloring each point by its corresponding digit label.
3. Compare the structure of the two plots (Autoencoder vs. PCA) and briefly discuss the differences (if any) you observe.

In [ ]:
# fit pca with 2 components on fitting set and transform reporting set
pca = PCA(n_components=2, random_state=42)
pca.fit(x_fit)
latents_pca = pca.transform(x_report)

# print explained variance ratio
var_comp1, var_comp2 = pca.explained_variance_ratio_
total_var = var_comp1 + var_comp2
print(f"PCA Component 1 Explained Variance: {var_comp1:.4f} ({var_comp1*100:.2f}%)")
print(f"PCA Component 2 Explained Variance: {var_comp2:.4f} ({var_comp2*100:.2f}%)")
print(f"PCA Total Explained Variance: {total_var:.4f} ({total_var*100:.2f}%)")

# side-by-side scatter plots comparing autoencoder and pca
_fig, _axes = plt.subplots(1, 2, figsize=(16, 6.5))

# autoencoder scatter
_sc1 = _axes[0].scatter(
    latents_autoencoder[:, 0],
    latents_autoencoder[:, 1],
    c=labels,
    cmap="tab10",
    alpha=0.5,
    s=8,
)
_axes[0].set_title("autoencoder 2d latent space (reporting set)", fontsize=11)
_axes[0].set_xlabel("latent dimension 1")
_axes[0].set_ylabel("latent dimension 2")
_axes[0].grid(True, alpha=0.3)

# pca scatter
_sc2 = _axes[1].scatter(
    latents_pca[:, 0],
    latents_pca[:, 1],
    c=labels,
    cmap="tab10",
    alpha=0.5,
    s=8,
)
_axes[1].set_title("pca 2d projection (reporting set)", fontsize=11)
_axes[1].set_xlabel("principal component 1")
_axes[1].set_ylabel("principal component 2")
_axes[1].grid(True, alpha=0.3)

_cbar_comp = _fig.colorbar(_sc2, ax=_axes, ticks=range(10), fraction=0.03, pad=0.04)
_cbar_comp.set_label("digit label", fontsize=10)
plt.suptitle("comparison of 2d representations: autoencoder vs. pca", fontsize=13)
plt.show()

### Comparison: Autoencoder vs. PCA

#### 1. Cluster structure and shape
The autoencoder forms curved trajectories with distinct branches for digits like 0 and 1, grouping points into directional bands via its ReLU activations. Linear PCA instead collapses all digits into a single dense elliptical cloud around the origin. Digits blend together across the central region with noticeably looser separation.

#### 2. Dimensionality reduction mechanism
PCA seeks two orthogonal directions that capture the maximum pixel variance. The first two principal components account for only 16.87% of the total pixel variance (9.74% on the first component and 7.13% on the second component). Because handwritten digits have curved, non-linear pixel dependencies, projecting onto two flat linear axes discards more than 83% of the signal. The autoencoder bends its coordinate system through non-linear layers, allowing the 2D bottleneck to preserve structural relationships that a linear projection flattens.

## 6. Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

Our main challenge was coordinating training setups and early stopping patience across different team members. Early on, one member ran Adam on CPU with small 64-sample batches and zero weight decay, which diverged from GPU runs using batch size 128 and 1e-5 weight decay. This produced differing loss trajectories and shifted latent scales during initial comparisons until we matched all settings. To resolve this, our group fixed random seed 42 and unified batch size to 128 across our training runs. For the next lab, we will write an explicit hyperparameter checklist before splitting code tasks among group members.